# Pipeline paso a paso: AgroNova (empresa ficticia)

Todos los datos son **sintéticos**. Este cuaderno recorre el flujo del repositorio: ERP simulado → Bronze → Silver → control → Gold → valor. Ejecuta las celdas en orden. Para entender el *porqué* de cada paso, lee `docs/01_caso_agronova.md`.

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
from agronova.fuentes import crear_erp_simulado
from agronova.ingesta import extraer_incremental, cargar_bronze
from agronova.silver import a_silver
from agronova.control import reconciliar
from agronova.contrato import validar_contrato
from agronova.gold import a_gold
from agronova.valor import calcular_valor

## 1. Fuente: un ERP simulado
Las líneas de orden de compra llegan con defectos típicos: texto con espacios, unidades escritas de varias formas y cantidades con punto de miles.

In [ ]:
erp = crear_erp_simulado()
crudo = extraer_incremental(erp, None)      # sin marca de agua = carga completa
crudo.head(5)

## 2. Bronze: copia fiel con trazabilidad
No se corrige nada. Se añaden el lote y la fecha de carga.

In [ ]:
bronze = cargar_bronze(pd.DataFrame(), crudo, "lote-001")
print(len(bronze), "filas en Bronze")
bronze[["id_oc", "proveedor", "unidad_txt", "cantidad_txt", "lote_carga"]].head(5)

## 3. Silver: limpiar, validar y separar lo rechazado

In [ ]:
silver, cuarentena = a_silver(bronze)
print("Silver:", len(silver), "| Cuarentena:", len(cuarentena))
print("Violaciones del contrato:", validar_contrato(silver))
silver[["id_oc", "id_proveedor", "unidad", "cantidad", "fecha_prometida", "fecha_recepcion"]].head(5)

## 4. El error que no hace fallar el pipeline
Una regla defectuosa trata la cantidad cero como inválida. Los conteos siguen cuadrando, pero lo delata el umbral de cuarentena.

In [ ]:
s_mal, c_mal = a_silver(bronze, regla_defectuosa=True)
reconciliar(bronze, s_mal, c_mal)

## 5. Gold: el KPI se define una sola vez
OTD correcto = líneas a tiempo ÷ **todas** las líneas vencidas. El OTD «ingenuo» ignora las que no llegaron.

In [ ]:
a_gold(silver)

## 6. Valor: estimado, no realizado
Todos los supuestos son ficticios.

In [ ]:
calcular_valor()

## Límites
Este cuaderno es demostrativo. Las pruebas usan datos sintéticos que construí para reproducir las cifras del caso: demuestran que el código hace lo que dice, no que el resultado sea válido en una empresa real. Consulta `docs/09_alcance_y_validacion.md`.